# Lab U3: Jacobian matrices and local linearization

**Unit:** Unit 3, Beyond Linear Maps  
**Role:** Required  
**Textbook sections:** Jacobian matrices; local linearization; chain rule as matrix multiplication  
**Core path:** symbolic Jacobian matrix computations, evaluating Jacobian matrices, actual versus local linear prediction, chain-rule shape checks  
**Assessment note:** This notebook is a supplement to the textbook, not a separate programming assignment. The main goal is to interpret computations, not to memorize code syntax.

This lab practices reading Jacobian matrix computations. The main comparison is

\[
F(\mathbf{a}+\mathbf{h}) \quad \text{versus} \quad F(\mathbf{a}) + J_F(\mathbf{a})\mathbf{h}.
\]

The first quantity is the actual nonlinear output. The second is the local linear prediction.


### Computational tools used in this lab

Before starting, review these parts of **Appendix A, NumPy and SymPy Quick Reference for the Labs**:

- Appendix A.2: NumPy arrays, vectors, matrices, and shapes
- Appendix A.4: Elementwise arithmetic versus linear algebra
- Appendix A.11: SymPy symbolic computation

The goal is to interpret the mathematical computation, not to memorize every command.

## Part 0. Setup and shape habits

Shape habit: Before reading a product such as `J_at_a @ h`, identify the shape of `J_at_a` and the length of `h`.

Predict before running:

1. What is the input dimension?
2. What is the output dimension?
3. Why is `J_example @ h_example` defined?
4. What is the shape of the output?


In [ ]:
import numpy as np
import sympy as sp

np.set_printoptions(precision=4, suppress=True)

J_example = np.array([[1.0, 2.0],
                      [0.0, -1.0],
                      [3.0, 4.0]])
h_example = np.array([0.1, -0.2])

J_example.shape, h_example.shape, J_example @ h_example


## Part 1. Core path: symbolic Jacobian matrix

Math reminder:

\[
F(x,y)=
\begin{bmatrix}
x^2y\\
xe^y
\end{bmatrix}.
\]

Predict before running:

1. How many input variables does \(F\) have?
2. How many output components does \(F\) have?
3. What should the shape of \(J\) be?


In [ ]:
x, y = sp.symbols("x y")

F = sp.Matrix([
    x**2 * y,
    x * sp.exp(y)
])

J = F.jacobian([x, y])
F, J


Interpretation check:

1. Which row corresponds to the second component of \(F\)?
2. Which column describes changing \(y\)?
3. How does this match the Unit 1 idea "rows measure; columns contribute"?


## Part 2. Evaluate a Jacobian matrix at a point

Math reminder:

\[
\mathbf{a}=\begin{bmatrix}1\\0\end{bmatrix}.
\]

Predict before running:

1. Which symbols should be replaced by the entries of \(\mathbf{a}\)?
2. Should \(J_F(\mathbf{a})\) still have the same shape as \(J_F(x,y)\)?


In [ ]:
a = np.array([1.0, 0.0])

J_at_a_sym = J.subs({x: a[0], y: a[1]})
J_at_a = np.array(J_at_a_sym, dtype=float)

J_at_a_sym, J_at_a, J_at_a.shape


Interpretation check:

1. What is \(J_F(\mathbf{a})\)?
2. What is its shape?
3. What does row 1 measure?
4. What does column 2 measure?


## Part 3. Actual output versus local linear prediction

Math reminder: compare the actual nonlinear output with the local linear prediction.

\[
F(\mathbf{a}+\mathbf{h}) \quad \text{versus} \quad F(\mathbf{a})+J_F(\mathbf{a})\mathbf{h}.
\]

Predict before running:

1. Which line below will compute the actual nonlinear output?
2. Which line below will compute the local linear prediction?


In [ ]:
def F_np(v):
    x_val, y_val = v
    return np.array([
        x_val**2 * y_val,
        x_val * np.exp(y_val)
    ], dtype=float)

h = np.array([0.1, -0.2])

actual = F_np(a + h)
linear = F_np(a) + J_at_a @ h
error = actual - linear

actual, linear, error


Interpretation check:

1. Which output is the actual nonlinear output?
2. Which output is the local linear prediction?
3. What does the error vector measure?
4. Is the prediction exact?


## Part 4. What happens when the input change gets smaller?

Predict before running: as the scale decreases, what should happen to the error norm?


In [ ]:
for scale in [1.0, 0.5, 0.25, 0.125]:
    h_scaled = scale * h
    actual_scaled = F_np(a + h_scaled)
    linear_scaled = F_np(a) + J_at_a @ h_scaled
    error_scaled = actual_scaled - linear_scaled
    print(scale, actual_scaled, linear_scaled, error_scaled, np.linalg.norm(error_scaled))


Interpretation check:

1. As the scale decreases, what happens to the error norm?
2. Why does this support the phrase "local linear approximation"?
3. Why would this not prove that the approximation is good far away from \(\mathbf{a}\)?


## Square-grid visualizations

A matrix sends grid lines to grid lines. A nonlinear map can bend grid lines.

We compare the linear shear

\[
T\left(\begin{bmatrix}x\\y\end{bmatrix}\right)
=
\begin{bmatrix}
x+y\\
y
\end{bmatrix}
\]

with the nonlinear shear

\[
F\left(\begin{bmatrix}x\\y\end{bmatrix}\right)
=
\begin{bmatrix}
x+y^2\\
y
\end{bmatrix}.
\]

The two maps agree on the four corners of the unit square. The square-grid visualization samples more points.


In [ ]:
import matplotlib.pyplot as plt

A_shear = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
])

def T(P):
    """Linear shear. Columns of P are input points."""
    return A_shear @ P

def F_grid(P):
    """Nonlinear shear. Columns of P are input points."""
    x = P[0, :]
    y = P[1, :]
    return np.vstack([x + y**2, y])

def square_grid(n=21, lo=0.0, hi=1.0):
    t = np.linspace(lo, hi, n)
    X, Y = np.meshgrid(t, t)
    P = np.vstack([X.ravel(), Y.ravel()])
    color = X.ravel() + 2 * Y.ravel()
    return P, color

def square_grid_lines(n=11, samples=150, lo=0.0, hi=1.0):
    t = np.linspace(lo, hi, samples)
    levels = np.linspace(lo, hi, n)

    lines = []
    for s in levels:
        lines.append(np.vstack([t, np.full_like(t, s)]))
        lines.append(np.vstack([np.full_like(t, s), t]))

    return lines

def plot_grid_map(map_fn, title, n_points=25, n_lines=11):
    P, color = square_grid(n_points)
    Q = map_fn(P)
    lines = square_grid_lines(n_lines)

    fig, ax = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)

    for L in lines:
        ax[0].plot(L[0, :], L[1, :], linewidth=0.6, alpha=0.5)
        M = map_fn(L)
        ax[1].plot(M[0, :], M[1, :], linewidth=0.6, alpha=0.5)

    sc = ax[0].scatter(P[0, :], P[1, :], c=color, cmap="viridis", s=12)
    ax[1].scatter(Q[0, :], Q[1, :], c=color, cmap="viridis", s=12)

    ax[0].set_title("Input grid")
    ax[1].set_title(title)

    for axis in ax:
        axis.set_aspect("equal", adjustable="box")
        axis.set_xlabel("$x$")
        axis.set_ylabel("$y$")

    fig.colorbar(sc, ax=ax, shrink=0.8, label="$x+2y$")
    return fig, ax

plot_grid_map(T, "Linear shear")
plot_grid_map(F_grid, "Nonlinear shear");


### Check yourself

1. Which map is a matrix map?
2. Which map contains the nonlinear operation?
3. Why do the four corners fail to detect the difference?
4. Which grid lines stay straight under the linear shear?
5. Which grid lines bend under the nonlinear shear?


## Local square-grid visualization

Now center a small square at

\[
\mathbf{a}
=
\begin{bmatrix}
0\\
1/2
\end{bmatrix}.
\]

Near this point, the nonlinear map \(F\) is approximated by

\[
F(\mathbf{a}+\mathbf{h})
\approx
F(\mathbf{a})+J_F(\mathbf{a})\mathbf{h}.
\]


In [ ]:
def JF_grid_at(a):
    """Jacobian matrix of F(x,y) = (x + y^2, y) at a."""
    a = np.asarray(a, dtype=float)
    return np.array([
        [1.0, 2.0 * a[1]],
        [0.0, 1.0],
    ])

def local_linearization(F, JF_at, a):
    """Return L_a(P) = F(a) + J_F(a)(P-a)."""
    a = np.asarray(a, dtype=float).reshape(2, 1)
    Fa = F(a)
    J = JF_at(a.ravel())

    def L(P):
        return Fa + J @ (P - a)

    return L

def centered_square(a, r=0.2, n=25):
    a = np.asarray(a, dtype=float)
    h = np.linspace(-r, r, n)
    H1, H2 = np.meshgrid(h, h)
    P = np.vstack([a[0] + H1.ravel(), a[1] + H2.ravel()])
    color = H1.ravel() + 2 * H2.ravel()
    return P, color

def centered_square_lines(a, r=0.2, n=9, samples=120):
    a = np.asarray(a, dtype=float)
    h = np.linspace(-r, r, samples)
    levels = np.linspace(-r, r, n)

    lines = []
    for s in levels:
        lines.append(np.vstack([a[0] + h, a[1] + np.full_like(h, s)]))
        lines.append(np.vstack([a[0] + np.full_like(h, s), a[1] + h]))

    return lines

def plot_local_comparison(F, JF_at, a=(0.0, 0.5), r=0.25):
    P, color = centered_square(a, r=r, n=25)
    L = local_linearization(F, JF_at, a)

    Fp = F(P)
    Lp = L(P)
    lines = centered_square_lines(a, r=r)

    fig, ax = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

    panels = [
        (P, "Input square"),
        (Fp, "Nonlinear image"),
        (Lp, "Local linear image"),
    ]

    for axis, (Pts, title) in zip(ax, panels):
        axis.scatter(Pts[0, :], Pts[1, :], c=color, cmap="viridis", s=10)
        axis.set_title(title)
        axis.set_aspect("equal", adjustable="box")
        axis.set_xlabel("$x$")
        axis.set_ylabel("$y$")

    for L0 in lines:
        ax[0].plot(L0[0, :], L0[1, :], linewidth=0.6, alpha=0.5)

        LF = F(L0)
        LL = L(L0)

        ax[1].plot(LF[0, :], LF[1, :], linewidth=0.6, alpha=0.5)
        ax[2].plot(LL[0, :], LL[1, :], linewidth=0.6, alpha=0.5)

    max_err = np.linalg.norm(Fp - Lp, axis=0).max()
    print(f"r = {r:.3f}, max ||F(p)-L_a(p)|| = {max_err:.5f}")

    return fig, ax

a = np.array([0.0, 0.5])

JF_grid_at(a)


In [ ]:
plot_local_comparison(F_grid, JF_grid_at, a=a, r=0.25)
plot_local_comparison(F_grid, JF_grid_at, a=a, r=0.10);


### Check yourself

1. What is \(J_F(\mathbf{a})\)?
2. Which Unit 1 matrix action does this Jacobian matrix match?
3. What changes when \(r\) decreases from \(0.25\) to \(0.10\)?
4. Is the local linearization describing the whole unit square or only a neighborhood of \(\mathbf{a}\)?
5. In the expression \(J_F(\mathbf{a})\mathbf{h}\), what does \(\mathbf{h}\) represent?


## Part 5. Scalar local linearization

Math reminder: for a scalar-valued function, the gradient gives the local linear part.

\[
f(x,y)=x^2+xy.
\]

Predict before running:

1. Is the output of \(f\) a scalar or a vector?
2. What shape should the gradient have at a point?


In [ ]:
f = x**2 + x*y
grad_f = sp.Matrix([sp.diff(f, x), sp.diff(f, y)])

a_scalar = np.array([1.0, 2.0])
h_scalar = np.array([0.1, -0.2])

grad_at_a = np.array(
    grad_f.subs({x: a_scalar[0], y: a_scalar[1]}),
    dtype=float
).reshape(2)

def f_np(v):
    x_val, y_val = v
    return x_val**2 + x_val*y_val

prediction = f_np(a_scalar) + grad_at_a @ h_scalar
actual_scalar = f_np(a_scalar + h_scalar)

grad_f, grad_at_a, prediction, actual_scalar, actual_scalar - prediction


Interpretation check:

1. Why is the gradient a vector but the output of \(f\) is a scalar?
2. Where does the dot product appear?
3. What does the error measure?
4. How does this connect to tangent planes?


## Part 6. Affine maps are exact local models

Math reminder: an affine map has the form `A @ v + b`. It is not usually linear as a function of `v`, but its change is controlled by the linear map `A`.

Predict before running: what should the error be?


In [ ]:
A = np.array([[2.0, -1.0],
              [0.0,  3.0]])
b = np.array([1.0, -2.0])

def affine_map(v):
    return A @ v + b

a_lin = np.array([1.0, 2.0])
h_lin = np.array([0.3, -0.4])

actual_affine = affine_map(a_lin + h_lin)
linear_prediction_affine = affine_map(a_lin) + A @ h_lin

actual_affine, linear_prediction_affine, actual_affine - linear_prediction_affine


Interpretation check:

1. Why is the error zero?
2. Is `v -> A @ v + b` linear or affine?
3. What is the linear map on input changes?
4. Why does the bias vector disappear when comparing changes?


## Part 7. Jacobian matrix columns

Math reminder: a column of a Jacobian matrix describes the predicted output change from changing one input coordinate.

Predict before running: what should `J_at_a @ e1` and `J_at_a @ e2` return?


In [ ]:
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])

J_at_a @ e1, J_at_a @ e2, J_at_a[:, 0], J_at_a[:, 1]


Interpretation check:

1. What does `J_at_a @ e1` return?
2. What does `J_at_a @ e2` return?
3. Why do these match the columns of `J_at_a`?
4. What does each column measure locally?


## Part 8. Locally forgotten directions

Extension connection: Unit 2 null-space language helps interpret a Jacobian matrix that sends a nonzero input change to zero.

Predict before running: what should happen to `J_forget @ h_forget`?


In [ ]:
J_forget = np.array([[1.0, 2.0],
                     [2.0, 4.0]])

h_forget = np.array([-2.0, 1.0])

J_forget @ h_forget


Interpretation check:

1. Why is `h_forget` a locally forgotten direction?
2. How does this connect to Unit 2 null spaces?
3. Does \(J_F(\mathbf{a})\mathbf{h}=\mathbf{0}\) prove that \(F(\mathbf{a}+\mathbf{h})=F(\mathbf{a})\) exactly?
4. What phrase should be used instead: exact equality or first-order prediction?


## Part 9. Chain rule as matrix multiplication

Math reminder:

\[
G:\mathbb R^2\to\mathbb R^3,
\qquad
F:\mathbb R^3\to\mathbb R^2.
\]

Before multiplying Jacobian matrices, check the input and output dimensions of each map.


In [ ]:
u, v = sp.symbols("u v")
s, t, r = sp.symbols("s t r")

G = sp.Matrix([
    u + v,
    u - v,
    u*v
])

F_outer = sp.Matrix([
    s**2 + t,
    sp.exp(r) + s*t
])

JG = G.jacobian([u, v])
JF_outer = F_outer.jacobian([s, t, r])

JG, JF_outer


In [ ]:
base = {u: 1, v: 2}

G_at_base = G.subs(base)

JF_at_G = JF_outer.subs({
    s: G_at_base[0],
    t: G_at_base[1],
    r: G_at_base[2]
})

JG_at_base = JG.subs(base)

J_comp = JF_at_G @ JG_at_base

G_at_base, JF_at_G, JG_at_base, J_comp


Interpretation check:

1. What is the shape of `JG`?
2. What is the shape of `JF_at_G`?
3. Which Jacobian matrix acts first on an input change?
4. Why is the product `JF_at_G @ JG_at_base`?
5. What shape should the composite Jacobian matrix have?


In [ ]:
# Debug check: uncomment to test the wrong order.
# JG_at_base @ JF_at_G


Debug check: Why should the reversed product fail or give the wrong interpretation?


## Part 10. Finite-difference check for the chain rule

Predict before running: for a small input change, should the chain-rule linear prediction be close to the actual composite output?


In [ ]:
def G_np(v_in):
    u_val, v_val = v_in
    return np.array([
        u_val + v_val,
        u_val - v_val,
        u_val * v_val
    ], dtype=float)

def F_outer_np(w_in):
    s_val, t_val, r_val = w_in
    return np.array([
        s_val**2 + t_val,
        np.exp(r_val) + s_val*t_val
    ], dtype=float)

def composite_np(v_in):
    return F_outer_np(G_np(v_in))

base_np = np.array([1.0, 2.0])
h_chain = np.array([0.01, -0.02])

J_comp_np = np.array(J_comp, dtype=float)

actual_chain = composite_np(base_np + h_chain)
linear_chain = composite_np(base_np) + J_comp_np @ h_chain

actual_chain, linear_chain, actual_chain - linear_chain


Interpretation check:

1. Which line computes the actual composite output?
2. Which line computes the local linear prediction?
3. Where does the chain rule matrix product appear?
4. Why is `h_chain` chosen small?


## Part 11. Reading a tiny sigmoid block

Use `u` for the hidden vector so it does not get confused with the input-change vector \(\mathbf{h}\).

Predict before running: which lines are affine, which line is nonlinear, and which line computes the Jacobian matrix at the input?


In [ ]:
def sigmoid(t):
    return 1 / (1 + np.exp(-t))

W1 = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

W2 = np.array([
    [1.0, 0.0, -0.5],
    [0.0, 1.0,  0.5],
])

b1 = np.zeros(3)
b2 = np.zeros(2)

x_input = np.array([0.0, 0.0])

s = W1 @ x_input + b1
u = sigmoid(s)
D = np.diag(u * (1 - u))
y = W2 @ u + b2
J_at_x = W2 @ D @ W1

s, u, y, J_at_x


Interpretation check:

1. Which lines are affine maps?
2. Which line contains the coordinatewise nonlinear step?
3. What do `s`, `u`, and `y` represent?
4. What does `D` store?
5. Which line computes the Jacobian matrix of the block at `x_input`?
6. What is the shape of `J_at_x`, and why?
7. Why is the whole block usually not one linear map?


## A square-grid visualization for a sigmoid block

A small smooth nonlinear block can also be tested with a square grid.

The block below has two inputs and two outputs, so we can draw its action on a grid:

\[
\mathbf{s}=W_1\mathbf{x}+\mathbf{b}_1,
\qquad
\mathbf{u}=\sigma(\mathbf{s}),
\qquad
N(\mathbf{x})=W_2\mathbf{u}+\mathbf{b}_2.
\]

The affine pieces use matrix multiplication and vector addition. The coordinatewise sigmoid step is nonlinear.

Because the sigmoid derivative changes with the input, the local matrix can change smoothly from point to point.


In [ ]:
def N_sigmoid(P):
    """Two-output sigmoid block. Columns of P are input points."""
    S = W1 @ P + b1.reshape(-1, 1)
    U = sigmoid(S)
    return W2 @ U + b2.reshape(-1, 1)

fig, ax = plot_grid_map(N_sigmoid, "Tiny sigmoid block", n_points=31, n_lines=13);


### Check yourself

1. Which lines are affine maps?
2. Which line is nonlinear?
3. What do the columns of the input grid matrix represent?
4. Why can the output grid bend or compress?
5. Where does the derivative matrix enter the local linear approximation?


## Reading the local matrix

For the sigmoid block, the diagonal matrix \(D\) stores the derivatives
\(\sigma'(s_i)=\sigma(s_i)(1-\sigma(s_i))\) at the current input.

At a base point \(\mathbf{a}\),

\[
J_N(\mathbf{a})=W_2D_{\mathbf{a}}W_1.
\]


In [ ]:
def sigmoid_data_at(a):
    """Return sigmoid inputs, hidden vector, derivative matrix, and local matrix."""
    a = np.asarray(a, dtype=float)
    s_at = W1 @ a + b1
    u_at = sigmoid(s_at)
    D_at = np.diag(u_at * (1 - u_at))
    J_at = W2 @ D_at @ W1
    return s_at, u_at, D_at, J_at

a = np.array([0.0, 0.0])
s_a, u_a, D_a, J_a = sigmoid_data_at(a)

s_a, u_a, J_a


At \(\mathbf{a}=(0,0)\), the sigmoid inputs are all zero and the hidden vector is \((1/2,1/2,1/2)\). The local matrix is

\[
\begin{bmatrix}
1/8&-1/8\\
1/8&3/8
\end{bmatrix}.
\]

This matrix predicts the first-order output change near \(\mathbf{a}\).


In [ ]:
c = np.array([0.75, 0.75])
s_c, u_c, D_c, J_c = sigmoid_data_at(c)

s_c, u_c, J_c


### Check yourself

1. At \(\mathbf{a}=(0,0)\), what does \(D_{\mathbf{a}}\) store?
2. Which input change does the first column of \(J_N(\mathbf{a})\) describe?
3. At \(\mathbf{c}=(0.75,0.75)\), why is \(D_{\mathbf{c}}\) different?
4. Why can the two local matrices differ?
5. Why does this example still fit the Unit 3 theme \(F(\mathbf{a}+\mathbf{h})\approx F(\mathbf{a})+J_F(\mathbf{a})\mathbf{h}\)?


## A square-grid visualization for softmax weights

Now the query vector moves across a square.

The keys and values are fixed:

\[
K=
\begin{bmatrix}
3&0\\
0&3\\
-3&-3
\end{bmatrix},
\qquad
V=
\begin{bmatrix}
1&0\\
0&1\\
0&0
\end{bmatrix}.
\]

For each query vector \(\mathbf{q}\), compute

\[
\mathbf{s}=K\mathbf{q},
\qquad
\boldsymbol{\alpha}=\operatorname{softmax}(\mathbf{s}),
\qquad
F(\mathbf{q})=V^T\boldsymbol{\alpha}.
\]

The map \(K\mathbf{q}\) is linear. The softmax step is nonlinear. The output is a weighted average of the rows of \(V\).


In [ ]:
K_attention = np.array([
    [3.0, 0.0],
    [0.0, 3.0],
    [-3.0, -3.0],
])

V_attention = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 0.0],
])

def softmax_columns(S):
    """Apply softmax separately to each column."""
    S_shifted = S - S.max(axis=0, keepdims=True)
    E = np.exp(S_shifted)
    return E / E.sum(axis=0, keepdims=True)

def F_attention(P):
    """Attention-style map. Columns of P are query vectors."""
    S = K_attention @ P
    Alpha = softmax_columns(S)
    return V_attention.T @ Alpha


In [ ]:
test_queries = np.array([
    [0.0, 1.0, 0.0, -1.0],
    [0.0, 0.0, 1.0, -1.0],
])

scores = K_attention @ test_queries
weights = softmax_columns(scores)
outputs = F_attention(test_queries)

np.round(scores, 3), np.round(weights, 3), np.round(outputs, 3)


### Check yourself

1. Which columns of `test_queries` are being tested?
2. Which line computes the score vectors?
3. Which line applies the nonlinear rule?
4. Why do the columns of `weights` each add to \(1\)?
5. Why are the columns of `outputs` weighted averages of the rows of \(V\)?


In [ ]:
def attention_query_grid(n=31, lo=-1.0, hi=1.0):
    t = np.linspace(lo, hi, n)
    X, Y = np.meshgrid(t, t)
    P = np.vstack([X.ravel(), Y.ravel()])
    color = X.ravel() + 2 * Y.ravel()
    return P, color

def attention_query_grid_lines(n=13, samples=150, lo=-1.0, hi=1.0):
    t = np.linspace(lo, hi, samples)
    levels = np.linspace(lo, hi, n)

    lines = []
    for s in levels:
        lines.append(np.vstack([t, np.full_like(t, s)]))
        lines.append(np.vstack([np.full_like(t, s), t]))

    return lines

def plot_attention_grid_map():
    P, color = attention_query_grid()
    Q = F_attention(P)
    lines = attention_query_grid_lines()

    fig, ax = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)

    for L in lines:
        ax[0].plot(L[0, :], L[1, :], linewidth=0.6, alpha=0.5)
        M = F_attention(L)
        ax[1].plot(M[0, :], M[1, :], linewidth=0.6, alpha=0.5)

    ax[0].scatter(P[0, :], P[1, :], c=color, cmap="viridis", s=10)
    ax[1].scatter(Q[0, :], Q[1, :], c=color, cmap="viridis", s=10)

    triangle = np.array([
        [1.0, 0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0, 0.0],
    ])
    ax[1].plot(triangle[0, :], triangle[1, :], linewidth=1.0)
    ax[1].scatter(V_attention[:, 0], V_attention[:, 1], s=35)

    ax[0].set_title("Query grid")
    ax[1].set_title("Softmax-weighted image")

    ax[0].set_xlabel("$q_1$")
    ax[0].set_ylabel("$q_2$")
    ax[1].set_xlabel("$F_1$")
    ax[1].set_ylabel("$F_2$")

    for axis in ax:
        axis.set_aspect("equal", adjustable="box")

    return fig, ax

plot_attention_grid_map();


### Check yourself

1. Where do the output points lie?
2. Why does the output stay inside the triangle formed by the three value vectors?
3. Why does the grid bend instead of staying a parallelogram grid?
4. Why does \(F(\mathbf{0})\ne \mathbf{0}\)?
5. Why does \(F(\mathbf{0})\ne \mathbf{0}\) prove that this map is not linear?


## Final synthesis

The square-grid examples used the same convention: columns are input points.

The examples showed three different kinds of nonlinear behavior.

- The nonlinear shear bends grid lines smoothly.
- The sigmoid block bends and compresses the grid because the sigmoid derivative changes with the input.
- The softmax-weighted map bends and compresses the grid because the weights depend on the input.

The Jacobian matrix gives the local matrix near one point when the map is differentiable there.

### Check yourself

1. What do the columns of the input grid matrix represent?
2. Why are four corners enough to describe a \(2\times 2\) matrix map, but not a general nonlinear map?
3. Which square-grid example bends grid lines smoothly?
4. Which square-grid example uses a diagonal derivative matrix for a coordinatewise nonlinear step?
5. Which square-grid example keeps outputs inside the triangle of value vectors?
6. What does \(J_F(\mathbf{a})\mathbf{h}\) predict?
7. Why is \(\mathbf{h}\), not \(\mathbf{x}\), the input to the local linear map?


## Part 12. Review code-reading bank

These are short checks for discussion or self-review. They are meant to be answered without writing long programs.

### Exam-style check

1. **Shape check.** In `J.shape`, which entry gives the output dimension and which gives the input dimension?
2. **Predicted change.** In `J_at_a @ h`, is the result an actual nonlinear output or a predicted output change?
3. **Actual output.** In `F_np(a + h)`, is this the actual nonlinear output or the local prediction?
4. **Affine versus linear.** In `F_np(a) + J_at_a @ h`, why is the expression affine in `a + h` but linear in `h`?
5. **Error vector.** What does `actual - linear` measure?
6. **Column reading.** In `J_at_a[:, 0]`, which coordinate direction does this column describe?
7. **Scalar local model.** In `grad_at_a @ h_scalar`, why does a dot product appear for scalar-valued functions?
8. **Chain-rule order.** In `JF_at_G @ JG_at_base`, which map is applied first?
9. **Wrong order.** What is wrong with trying `JG_at_base @ JF_at_G`?
10. **Affine exactness.** Why is `affine_map(a_lin+h_lin) - (affine_map(a_lin)+A @ h_lin)` the zero vector?
11. **Forgotten direction.** What does `J_forget @ h_forget` being zero mean locally?
12. **Tiny sigmoid block.** Is `W1 @ x_input + b1` linear or affine as a function of `x_input`? Which line computes the Jacobian matrix at `x_input`?

### Answer sketches

1. For a Jacobian matrix with shape `(m, n)`, `m` is the output dimension and `n` is the input dimension.
2. A predicted output change.
3. The actual nonlinear output.
4. The base value is a shift; the input-change part `J_at_a @ h` is linear in `h`.
5. The difference between the actual nonlinear output and the local linear prediction.
6. The first input-coordinate direction.
7. A scalar-valued local model sends an input change to one number, so the gradient dots with the input change.
8. `JG_at_base` acts first on the input change; then `JF_at_G` acts on the resulting change.
9. The dimensions do not match the order of composition, and the interpretation is reversed.
10. For an affine map, changes are exactly controlled by the linear part `A`.
11. The first-order prediction says that direction produces no output change.
12. Affine, because the bias vector is added after multiplication. The line `J_at_x = W2 @ D @ W1` computes the Jacobian matrix at `x_input`.
